In [130]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import nibabel as nib
from nibabel.processing import resample_from_to
import numpy as np
import pandas as pd


def _load_canonical_3d(path) -> nib.Nifti1Image:
    """Load a 3D NIfTI image in canonical RAS orientation."""

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

    image = nib.as_closest_canonical(
        nib.load(str(path))
    )

    if len(image.shape) != 3:
        raise ValueError(
            f"Expected a 3D image, but {path.name} "
            f"has shape {image.shape}."
        )

    return image


def _check_axis_aligned(image: nib.Nifti1Image) -> None:

    matrix = np.asarray(
        image.affine[:3, :3],
        dtype=float,
    )

    off_diagonal = matrix.copy()
    np.fill_diagonal(off_diagonal, 0.0)

    if not np.allclose(
        off_diagonal,
        0.0,
        atol=1e-5,
    ):
        raise ValueError(
            "The image grid is oblique. Splitting the mask "
            "using physical x = 0 is not reliable."
        )


def _physical_axis_coordinates(
    image: nib.Nifti1Image,
    axis: int,
) -> np.ndarray:

    number_of_voxels = image.shape[axis]

    indices = np.zeros(
        (number_of_voxels, 3),
        dtype=float,
    )

    indices[:, axis] = np.arange(
        number_of_voxels,
        dtype=float,
    )

    physical_points = nib.affines.apply_affine(
        image.affine,
        indices,
    )

    return physical_points[:, axis]


def create_lc_geometry_tables(
    masks: dict[str, str],
    *,
    threshold: float = 0.0,
    include_empty_slices: bool = False,
    include_bilateral_summary: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    if not masks:
        raise ValueError("At least one mask must be provided.")

    reference_name = next(iter(masks))
    reference_image = _load_canonical_3d(
        masks[reference_name]
    )

    _check_axis_aligned(reference_image)

    reference_shape = reference_image.shape
    reference_affine = reference_image.affine

    voxel_sizes = nib.affines.voxel_sizes(
        reference_affine
    )

    spacing_x_mm = float(voxel_sizes[0])
    spacing_y_mm = float(voxel_sizes[1])
    spacing_z_mm = float(voxel_sizes[2])

    voxel_area_mm2 = (
        spacing_x_mm
        * spacing_y_mm
    )

    voxel_volume_mm3 = (
        spacing_x_mm
        * spacing_y_mm
        * spacing_z_mm
    )

    x_coordinates_mm = _physical_axis_coordinates(
        reference_image,
        axis=0,
    )

    left_half = np.broadcast_to(
        x_coordinates_mm[:, None, None] < 0.0,
        reference_shape,
    )

    right_half = np.broadcast_to(
        x_coordinates_mm[:, None, None] > 0.0,
        reference_shape,
    )

    loaded_masks = {}

    for mask_name, mask_path in masks.items():
        mask_image = _load_canonical_3d(mask_path)
        _check_axis_aligned(mask_image)

        same_grid = (
            mask_image.shape == reference_shape
            and np.allclose(
                mask_image.affine,
                reference_affine,
                atol=1e-4,
            )
        )

        if not same_grid:
            raise ValueError(
                f"Mask-grid mismatch for {mask_name!r}.\n"
                f"Reference shape: {reference_shape}\n"
                f"Mask shape:      {mask_image.shape}\n"
                "Masks are not automatically resampled because "
                "that would change voxel counts and volumes."
            )

        mask_data = np.asanyarray(
            mask_image.dataobj
        )

        full_mask = (
            np.isfinite(mask_data)
            & (mask_data > threshold)
        )

        left_mask = full_mask & left_half
        right_mask = full_mask & right_half

        loaded_masks[mask_name] = {
            "path": str(Path(mask_path)),
            "left": left_mask,
            "right": right_mask,
        }

    per_z_rows = []
    summary_rows = []

    for mask_name, mask_information in loaded_masks.items():

        for hemisphere in ("left", "right"):
            hemisphere_mask = mask_information[
                hemisphere
            ]

            voxel_counts_per_z = (
                hemisphere_mask
                .sum(axis=(0, 1))
                .astype(int)
            )

            for z_index, voxel_count in enumerate(
                voxel_counts_per_z
            ):
                if (
                    not include_empty_slices
                    and voxel_count == 0
                ):
                    continue

                per_z_rows.append(
                    {
                        "mask": mask_name,
                        "mask_path": mask_information["path"],
                        "hemisphere": hemisphere,
                        "z_index": int(z_index),
                        "voxel_count": int(voxel_count),
                        "area_mm2": float(
                            voxel_count
                            * voxel_area_mm2
                        ),
                        # Volume contribution of this z-slice.
                        "volume_mm3": float(
                            voxel_count
                            * voxel_volume_mm3
                        ),
                    }
                )

            total_voxels = int(
                hemisphere_mask.sum()
            )

            nonempty_z_indices = np.flatnonzero(
                voxel_counts_per_z > 0
            )

            summary_rows.append(
                {
                    "mask": mask_name,
                    "mask_path": mask_information["path"],
                    "hemisphere": hemisphere,
                    "total_voxels": total_voxels,
                    "total_volume_mm3": float(
                        total_voxels
                        * voxel_volume_mm3
                    ),
                    "number_of_nonempty_z_slices": int(
                        len(nonempty_z_indices)
                    ),
                    "first_nonempty_z_index": (
                        int(nonempty_z_indices[0])
                        if len(nonempty_z_indices) > 0
                        else np.nan
                    ),
                    "last_nonempty_z_index": (
                        int(nonempty_z_indices[-1])
                        if len(nonempty_z_indices) > 0
                        else np.nan
                    ),
                    "spacing_x_mm": spacing_x_mm,
                    "spacing_y_mm": spacing_y_mm,
                    "spacing_z_mm": spacing_z_mm,
                    "voxel_area_mm2": voxel_area_mm2,
                    "voxel_volume_mm3": voxel_volume_mm3,
                }
            )

        if include_bilateral_summary:
            bilateral_mask = (
                mask_information["left"]
                | mask_information["right"]
            )

            bilateral_voxels_per_z = (
                bilateral_mask
                .sum(axis=(0, 1))
                .astype(int)
            )

            bilateral_nonempty_z = np.flatnonzero(
                bilateral_voxels_per_z > 0
            )

            bilateral_total_voxels = int(
                bilateral_mask.sum()
            )

            summary_rows.append(
                {
                    "mask": mask_name,
                    "mask_path": mask_information["path"],
                    "hemisphere": "bilateral",
                    "total_voxels": bilateral_total_voxels,
                    "total_volume_mm3": float(
                        bilateral_total_voxels
                        * voxel_volume_mm3
                    ),
                    "number_of_nonempty_z_slices": int(
                        len(bilateral_nonempty_z)
                    ),
                    "first_nonempty_z_index": (
                        int(bilateral_nonempty_z[0])
                        if len(bilateral_nonempty_z) > 0
                        else np.nan
                    ),
                    "last_nonempty_z_index": (
                        int(bilateral_nonempty_z[-1])
                        if len(bilateral_nonempty_z) > 0
                        else np.nan
                    ),
                    "spacing_x_mm": spacing_x_mm,
                    "spacing_y_mm": spacing_y_mm,
                    "spacing_z_mm": spacing_z_mm,
                    "voxel_area_mm2": voxel_area_mm2,
                    "voxel_volume_mm3": voxel_volume_mm3,
                }
            )

    per_z_df = pd.DataFrame(per_z_rows)

    if not per_z_df.empty:
        per_z_df = (
            per_z_df
            .sort_values(
                [
                    "mask",
                    "hemisphere",
                    "z_index",
                ]
            )
            .reset_index(drop=True)
        )

    summary_df = (
        pd.DataFrame(summary_rows)
        .sort_values(
            [
                "mask",
                "hemisphere",
            ]
        )
        .reset_index(drop=True)
    )

    return per_z_df, summary_df

def plot_both_masks_from_row(
    per_z_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    row_number: int,
    tse_path,
    *,
    original_mask_name: str = "Original LC mask",
    refined_mask_name: str = "Refined FT cluster",
    original_alpha: float = 0.5,
    padding_mm: float = 5.0,
    figsize: tuple[float, float] = (9, 8),
    save_path=None,
) -> tuple[pd.DataFrame, plt.Figure, plt.Axes]:

    required_columns = {
        "mask",
        "mask_path",
        "hemisphere",
        "z_index",
        "voxel_count",
        "area_mm2",
        "volume_mm3",
    }

    missing_columns = required_columns.difference(
        per_z_df.columns
    )

    if missing_columns:
        raise KeyError(
            "per_z_df is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if not 0 <= row_number < len(per_z_df):
        raise IndexError(
            f"row_number must be between 0 and "
            f"{len(per_z_df) - 1}; received {row_number}."
        )

    if not 0.0 <= original_alpha <= 1.0:
        raise ValueError(
            "original_alpha must be between 0 and 1."
        )

    selected_row = per_z_df.iloc[row_number]

    hemisphere = str(
        selected_row["hemisphere"]
    ).lower()

    z_index = int(
        selected_row["z_index"]
    )

    if hemisphere not in {"left", "right"}:
        raise ValueError(
            "The selected row must represent the left "
            "or right hemisphere."
        )

    mask_names = [
        original_mask_name,
        refined_mask_name,
    ]

    mask_paths = {}

    for mask_name in mask_names:
        path_matches = per_z_df.loc[
            per_z_df["mask"] == mask_name,
            "mask_path",
        ]

        if path_matches.empty:
            raise ValueError(
                f"No path was found for {mask_name!r}."
            )

        mask_paths[mask_name] = Path(
            path_matches.iloc[0]
        )

    reference_image = _load_canonical_3d(
        mask_paths[original_mask_name]
    )

    _check_axis_aligned(reference_image)

    reference_shape = reference_image.shape
    reference_affine = reference_image.affine

    if not 0 <= z_index < reference_shape[2]:
        raise IndexError(
            f"z_index {z_index} is outside the valid range "
            f"0–{reference_shape[2] - 1}."
        )

    voxel_sizes = nib.affines.voxel_sizes(
        reference_affine
    )

    spacing_x_mm = float(voxel_sizes[0])
    spacing_y_mm = float(voxel_sizes[1])
    spacing_z_mm = float(voxel_sizes[2])

    voxel_area_mm2 = (
        spacing_x_mm
        * spacing_y_mm
    )

    voxel_volume_mm3 = (
        spacing_x_mm
        * spacing_y_mm
        * spacing_z_mm
    )

    x_coordinates_mm = _physical_axis_coordinates(
        reference_image,
        axis=0,
    )

    y_coordinates_mm = _physical_axis_coordinates(
        reference_image,
        axis=1,
    )

    if hemisphere == "left":
        hemisphere_condition = (
            x_coordinates_mm[:, None, None] < 0.0
        )
    else:
        hemisphere_condition = (
            x_coordinates_mm[:, None, None] > 0.0
        )

    hemisphere_half = np.broadcast_to(
        hemisphere_condition,
        reference_shape,
    )

    loaded_masks = {}

    for mask_name, mask_path in mask_paths.items():
        mask_image = _load_canonical_3d(
            mask_path
        )

        same_grid = (
            mask_image.shape == reference_shape
            and np.allclose(
                mask_image.affine,
                reference_affine,
                atol=1e-4,
            )
        )

        if not same_grid:
            raise ValueError(
                f"{mask_name!r} does not share the "
                "original LC-mask grid."
            )

        mask_data = np.asanyarray(
            mask_image.dataobj
        )

        loaded_masks[mask_name] = (
            np.isfinite(mask_data)
            & (mask_data > 0)
            & hemisphere_half
        )

    # Load and, if needed, resample the TSE only for visualization.
    tse_image = _load_canonical_3d(
        tse_path
    )

    same_tse_grid = (
        tse_image.shape == reference_shape
        and np.allclose(
            tse_image.affine,
            reference_affine,
            atol=1e-4,
        )
    )

    if not same_tse_grid:
        print(
            "TSE and masks are on different grids.\n"
            "Resampling only the TSE to the mask grid "
            "for visualization."
        )

        tse_image = resample_from_to(
            tse_image,
            (
                reference_shape,
                reference_affine,
            ),
            order=1,
        )

    tse_data = tse_image.get_fdata(
        dtype=np.float32
    )

    tse_slice = tse_data[:, :, z_index]

    original_slice = loaded_masks[
        original_mask_name
    ][:, :, z_index]

    refined_slice = loaded_masks[
        refined_mask_name
    ][:, :, z_index]

    # Build the table shown beneath the figure.
    comparison_rows = []

    for mask_name in mask_names:
        mask_3d = loaded_masks[mask_name]
        mask_slice = mask_3d[:, :, z_index]

        voxel_count = int(
            mask_slice.sum()
        )

        area_mm2 = float(
            voxel_count
            * voxel_area_mm2
        )

        volume_mm3 = float(
            voxel_count
            * voxel_volume_mm3
        )

        summary_match = summary_df.loc[
            (summary_df["mask"] == mask_name)
            & (summary_df["hemisphere"] == hemisphere)
        ]

        if summary_match.empty:
            total_mask_voxels = int(
                mask_3d.sum()
            )

            total_mask_volume_mm3 = float(
                total_mask_voxels
                * voxel_volume_mm3
            )

            total_mask_z_slices = int(
                np.count_nonzero(
                    mask_3d.any(axis=(0, 1))
                )
            )

        else:
            summary_row = summary_match.iloc[0]

            total_mask_voxels = int(
                summary_row["total_voxels"]
            )

            total_mask_volume_mm3 = float(
                summary_row["total_volume_mm3"]
            )

            total_mask_z_slices = int(
                summary_row[
                    "number_of_nonempty_z_slices"
                ]
            )

        comparison_rows.append(
            {
                "mask": mask_name,
                "hemisphere": hemisphere,
                "z_index": z_index,
                "voxel_count": voxel_count,
                "area_mm2": area_mm2,
                "volume_mm3": volume_mm3,
                "total_mask_voxels": total_mask_voxels,
                "total_mask_volume_mm3": (
                    total_mask_volume_mm3
                ),
                "total_mask_z_slices": (
                    total_mask_z_slices
                ),
            }
        )

    comparison_df = pd.DataFrame(
        comparison_rows
    )

    # Robust MRI intensity scaling.
    valid_tse_values = tse_data[
        np.isfinite(tse_data)
        & ~np.isclose(tse_data, 0.0)
    ]

    if valid_tse_values.size:
        display_vmin, display_vmax = np.percentile(
            valid_tse_values,
            [1.0, 99.0],
        )
    else:
        display_vmin = None
        display_vmax = None

    image_extent = [
        float(
            x_coordinates_mm.min()
            - spacing_x_mm / 2.0
        ),
        float(
            x_coordinates_mm.max()
            + spacing_x_mm / 2.0
        ),
        float(
            y_coordinates_mm.min()
            - spacing_y_mm / 2.0
        ),
        float(
            y_coordinates_mm.max()
            + spacing_y_mm / 2.0
        ),
    ]

    figure, axis = plt.subplots(
        figsize=figsize
    )

    axis.imshow(
        tse_slice.T,
        origin="lower",
        extent=image_extent,
        cmap="gray",
        vmin=display_vmin,
        vmax=display_vmax,
        interpolation="nearest",
    )

    # Original mask: filled transparent overlay.
    if original_slice.any():
        original_overlay = np.ma.masked_where(
            ~original_slice.T,
            original_slice.T.astype(float),
        )

        axis.imshow(
            original_overlay,
            origin="lower",
            extent=image_extent,
            interpolation="nearest",
            cmap=ListedColormap(["cyan"]),
            vmin=0.0,
            vmax=1.0,
            alpha=original_alpha,
        )

    # Refined FT mask: contour line.
    if refined_slice.any():
        axis.contour(
            x_coordinates_mm,
            y_coordinates_mm,
            refined_slice.T.astype(float),
            levels=[0.5],
            linewidths=2.5,
            linestyles="solid",
            colors="magenta",
        )

    # Zoom around both masks.
    union_slice = original_slice | refined_slice

    if union_slice.any():
        mask_coordinates = np.argwhere(
            union_slice
        )

        selected_x = x_coordinates_mm[
            mask_coordinates[:, 0]
        ]

        selected_y = y_coordinates_mm[
            mask_coordinates[:, 1]
        ]

        axis.set_xlim(
            float(
                selected_x.min()
                - padding_mm
            ),
            float(
                selected_x.max()
                + padding_mm
            ),
        )

        axis.set_ylim(
            float(
                selected_y.min()
                - padding_mm
            ),
            float(
                selected_y.max()
                + padding_mm
            ),
        )

    legend_handles = [
        Patch(
            facecolor="cyan",
            alpha=original_alpha,
            label=original_mask_name,
        ),
        Line2D(
            [0],
            [0],
            color="magenta",
            linewidth=2.5,
            label=refined_mask_name,
        ),
    ]

    axis.legend(
        handles=legend_handles,
        loc="upper right",
    )

    axis.set_title(
        f"Original and refined LC masks — "
        f"{hemisphere} LC\n"
        f"Dataframe row {row_number}; "
        f"z index {z_index}"
    )

    axis.set_xlabel("x (mm)")
    axis.set_ylabel("y (mm)")
    axis.set_aspect("equal")

    figure.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        figure.savefig(
            save_path,
            dpi=200,
            bbox_inches="tight",
        )

        print(f"Saved figure: {save_path}")

    plt.show()

    print(
        f"\nMeasurements for dataframe row {row_number}: "
        f"{hemisphere} LC, z index {z_index}"
    )

    try:
        from IPython.display import display

        display(comparison_df)

    except ImportError:
        print(
            comparison_df.to_string(index=False)
        )

    return comparison_df, figure, axis

In [228]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd


def compute_paper_lc_contrast(
    tse_path,
    lc_mask_path,
    reference_mask_path,
    *,
    return_per_slice=False,
):

    #CONTRASTT = (peak_LC - peak_reference) / peak_reference

    tse_path = Path(tse_path)
    lc_mask_path = Path(lc_mask_path)
    reference_mask_path = Path(reference_mask_path)

    for name, path in {
        "TSE": tse_path,
        "LC mask": lc_mask_path,
        "Reference mask": reference_mask_path,
    }.items():
        if not path.exists():
            raise FileNotFoundError(f"{name} does not exist: {path}")

    tse_img = nib.load(str(tse_path))
    lc_img = nib.load(str(lc_mask_path))
    reference_img = nib.load(str(reference_mask_path))

    for name, image in {
        "LC mask": lc_img,
        "Reference mask": reference_img,
    }.items():
        if image.shape != tse_img.shape:
            raise ValueError(
                f"{name} shape {image.shape} does not match "
                f"TSE shape {tse_img.shape}."
            )

        if not np.allclose(image.affine, tse_img.affine, atol=1e-4):
            raise ValueError(
                f"{name} does not share the TSE affine/grid."
            )

    tse = tse_img.get_fdata(dtype=np.float64)
    lc = lc_img.get_fdata(dtype=np.float64)
    reference = reference_img.get_fdata(dtype=np.float64)

    lc_positive = np.isfinite(lc) & (lc > 0)
    positive_values = np.unique(lc[lc_positive])

    # Refined Step-8 mask: label 1=left and 2=right.
    if (
        np.any(np.isclose(positive_values, 1))
        and np.any(np.isclose(positive_values, 2))
        and len(positive_values) <= 2
    ):
        left_lc = np.isclose(lc, 1)
        right_lc = np.isclose(lc, 2)

    else:
        # Binary mask: split using physical x=0.
        nx, ny, nz = lc.shape
        affine = np.asarray(lc_img.affine, dtype=float)

        i = np.arange(nx)[:, None, None]
        j = np.arange(ny)[None, :, None]
        k = np.arange(nz)[None, None, :]

        x_mm = (
            affine[0, 0] * i
            + affine[0, 1] * j
            + affine[0, 2] * k
            + affine[0, 3]
        )

        left_lc = lc_positive & (x_mm < 0)
        right_lc = lc_positive & (x_mm > 0)

    reference_mask = (
        np.isfinite(reference)
        & (reference > 0)
    )

    valid_tse = (
        np.isfinite(tse)
        & ~np.isclose(tse, 0)
    )

    rows = []


## for each z_index i compute separately the lc contrast
    for z_index in range(tse.shape[2]):
        tse_slice = tse[:, :, z_index]
        valid_slice = valid_tse[:, :, z_index]

        left_slice = (
            left_lc[:, :, z_index]
            & valid_slice
        )

        right_slice = (
            right_lc[:, :, z_index]
            & valid_slice
        )

        reference_slice = (
            reference_mask[:, :, z_index]
            & valid_slice
        )

        if (
            not left_slice.any()
            or not right_slice.any()
            or not reference_slice.any()
        ):
            continue

        left_peak = float(
            np.max(tse_slice[left_slice])
        )

        right_peak = float(
            np.max(tse_slice[right_slice])
        )

        reference_peak = float(
            np.max(tse_slice[reference_slice])
        )

        if np.isclose(reference_peak, 0):
            continue

        left_contrast = (
            left_peak - reference_peak
        ) / reference_peak

        right_contrast = (
            right_peak - reference_peak
        ) / reference_peak

        mean_lr_contrast = float(
            np.mean(
                [
                    left_contrast,
                    right_contrast,
                ]
            )
        )

        rows.append(
            {
                "z_index": z_index,
                "left_peak_intensity": left_peak,
                "right_peak_intensity": right_peak,
                "reference_peak_intensity": reference_peak,
                "left_contrast": left_contrast,
                "right_contrast": right_contrast,
                "mean_lr_contrast": mean_lr_contrast,
                "mean_lr_contrast_percent": (
                    mean_lr_contrast * 100
                ),
            }
        )

    per_slice_df = pd.DataFrame(rows)

    if per_slice_df.empty:
        raise RuntimeError(
            "No valid slice contained left LC, right LC, "
            "and reference-mask voxels."
        )

    peak_row = per_slice_df.loc[
        per_slice_df["mean_lr_contrast"].idxmax()
    ]

    final_contrast = float(
        peak_row["mean_lr_contrast"]
    )

    #print(f"LC mask: {lc_mask_path.name}")
    print(f"Peak z-index: {int(peak_row['z_index'])}")
    print(f"Left contrast:  {peak_row['left_contrast']:.4f}")
    print(f"Right contrast: {peak_row['right_contrast']:.4f}")
    print(
        f"Final LC contrast: {final_contrast:.4f} "
        f"({final_contrast * 100:.2f}%)"
    )

    if return_per_slice:
        return final_contrast, per_slice_df

    return per_slice_df

In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd


def compute_mean_lc_contrast(
    tse_path,
    lc_mask_path,
    reference_mask_path,
    *,
    return_per_slice=False,
    reference_max=True,
):

    tse_path = Path(tse_path)
    lc_mask_path = Path(lc_mask_path)
    reference_mask_path = Path(reference_mask_path)

    for name, path in {
        "TSE": tse_path,
        "LC mask": lc_mask_path,
        "Reference mask": reference_mask_path,
    }.items():
        if not path.exists():
            raise FileNotFoundError(
                f"{name} does not exist: {path}"
            )

    tse_img = nib.load(str(tse_path))
    lc_img = nib.load(str(lc_mask_path))
    reference_img = nib.load(str(reference_mask_path))

    # All images must have the same voxel grid.
    for name, image in {
        "LC mask": lc_img,
        "Reference mask": reference_img,
    }.items():
        if image.shape != tse_img.shape:
            raise ValueError(
                f"{name} shape {image.shape} does not match "
                f"TSE shape {tse_img.shape}."
            )

        if not np.allclose(
            image.affine,
            tse_img.affine,
            atol=1e-4,
        ):
            raise ValueError(
                f"{name} does not share the TSE affine/grid."
            )

    tse = tse_img.get_fdata(dtype=np.float64)
    lc = lc_img.get_fdata(dtype=np.float64)
    reference = reference_img.get_fdata(dtype=np.float64)

    lc_positive = (
        np.isfinite(lc)
        & (lc > 0)
    )

    positive_values = np.unique(
        lc[lc_positive]
    )

    # Labelled refined mask:
    # 1 = left LC, 2 = right LC.
    if (
        np.any(np.isclose(positive_values, 1))
        and np.any(np.isclose(positive_values, 2))
        and len(positive_values) <= 2
    ):
        left_lc = np.isclose(lc, 1)
        right_lc = np.isclose(lc, 2)

    else:
        # Binary bilateral mask:
        # split according to physical x-coordinate.
        nx, ny, nz = lc.shape
        affine = np.asarray(
            lc_img.affine,
            dtype=float,
        )

        i = np.arange(nx)[:, None, None]
        j = np.arange(ny)[None, :, None]
        k = np.arange(nz)[None, None, :]

        x_mm = (
            affine[0, 0] * i
            + affine[0, 1] * j
            + affine[0, 2] * k
            + affine[0, 3]
        )

        left_lc = (
            lc_positive
            & (x_mm < 0)
        )

        right_lc = (
            lc_positive
            & (x_mm > 0)
        )

    reference_mask = (
        np.isfinite(reference)
        & (reference > 0)
    )

    valid_tse = (
        np.isfinite(tse)
        & ~np.isclose(tse, 0)
    )

    rows = []

    for z_index in range(tse.shape[2]):

        tse_slice = tse[:, :, z_index]
        valid_slice = valid_tse[:, :, z_index]

        left_slice = (
            left_lc[:, :, z_index]
            & valid_slice
        )

        right_slice = (
            right_lc[:, :, z_index]
            & valid_slice
        )

        reference_slice = (
            reference_mask[:, :, z_index]
            & valid_slice
        )

        # Bilateral contrast requires both LC sides
        # and reference voxels on the same slice.
        if (
            not left_slice.any()
            or not right_slice.any()
            or not reference_slice.any()
        ):
            continue

        # Mean of all LC voxels on this side and slice.
        left_mean = float(
            np.mean(tse_slice[left_slice])
        )

        right_mean = float(
            np.mean(tse_slice[right_slice])
        )

        if reference_max:
            reference_peak = float(
                np.max(tse_slice[reference_slice])
            )
        else:
            reference_peak = float(
                np.mean(tse_slice[reference_slice])
            )

        if np.isclose(reference_peak, 0):
            continue

        left_contrast = (
            left_mean - reference_peak
        ) / reference_peak

        right_contrast = (
            right_mean - reference_peak
        ) / reference_peak

        mean_lr_contrast = float(
            np.mean(
                [
                    left_contrast,
                    right_contrast,
                ]
            )
        )

        rows.append(
            {
                "z_index": int(z_index),

                "left_lc_voxels": int(
                    left_slice.sum()
                ),
                "right_lc_voxels": int(
                    right_slice.sum()
                ),
                "reference_voxels": int(
                    reference_slice.sum()
                ),

                "left_mean_intensity": left_mean,
                "right_mean_intensity": right_mean,
                "reference_peak_intensity": reference_peak,

                "left_contrast": left_contrast,
                "right_contrast": right_contrast,
                "mean_lr_contrast": mean_lr_contrast,

                "left_contrast_percent": (
                    left_contrast * 100
                ),
                "right_contrast_percent": (
                    right_contrast * 100
                ),
                "mean_lr_contrast_percent": (
                    mean_lr_contrast * 100
                ),
            }
        )

    per_slice_df = pd.DataFrame(rows)

    if per_slice_df.empty:
        raise RuntimeError(
            "No valid slice contained left LC, right LC, "
            "and reference-mask voxels."
        )

    # Select the highest bilateral contrast across z-slices.
    peak_row = per_slice_df.loc[
        per_slice_df[
            "mean_lr_contrast"
        ].idxmax()
    ]

    final_contrast = float(
        peak_row["mean_lr_contrast"]
    )

    print(f"Peak z-index: {int(peak_row['z_index'])}")

    print(
        f"Left mean intensity: "
        f"{peak_row['left_mean_intensity']:.4f}"
    )

    print(
        f"Right mean intensity: "
        f"{peak_row['right_mean_intensity']:.4f}"
    )

    print(
        f"Reference peak intensity: "
        f"{peak_row['reference_peak_intensity']:.4f}"
    )

    print(
        f"Left contrast:  "
        f"{peak_row['left_contrast']:.4f} "
        f"({peak_row['left_contrast_percent']:.2f}%)"
    )

    print(
        f"Right contrast: "
        f"{peak_row['right_contrast']:.4f} "
        f"({peak_row['right_contrast_percent']:.2f}%)"
    )

    print(
        f"Final mean LC contrast: "
        f"{final_contrast:.4f} "
        f"({final_contrast * 100:.2f}%)"
    )

    if return_per_slice:
        return final_contrast, per_slice_df

    return final_contrast

In [218]:
TSE_IMAGE = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed/step6_tse_mni/sub002/ses001/tse_in_MNI_brainstem_0p5mm.nii.gz"
LC_MASK = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed/step7_masks_grid/sub002/ses001/LC_mask_grid.nii.gz"
DPT_MASK = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed/step7_masks_grid/sub002/ses001/DPT_mask_grid.nii.gz"
LC_MASK_REFINED = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed/step8_contrast/sub002/ses001/lc_ft_cluster_mask.nii.gz"

In [208]:
TSE_IMAGE = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj017/step6_tse_mni/sub017/ses001/tse_in_MNI_brainstem_0p5mm.nii.gz"
LC_MASK = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj017/step7_masks_grid/sub017/ses001/LC_mask_grid.nii.gz"
DPT_MASK = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj017/step7_masks_grid/sub017/ses001/DPT_mask_grid.nii.gz"
LC_MASK_REFINED = "/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj017/step8_contrast/sub017/ses001/lc_ft_cluster_mask.nii.gz"

In [219]:
per_z_df, df = create_lc_geometry_tables(
    masks={
        "Original LC mask": LC_MASK,
        "Refined FT cluster": LC_MASK_REFINED,
    }
)

In [221]:
per_z_df[per_z_df["z_index"] == 94] 

,mask,mask_path,hemisphere,z_index,voxel_count,area_mm2,volume_mm3
10,Original LC mask,/home/maria/Documents/data/hiwi_sample/tmp-mar...,left,94,19,4.75,2.375
33,Original LC mask,/home/maria/Documents/data/hiwi_sample/tmp-mar...,right,94,17,4.25,2.125
61,Refined FT cluster,/home/maria/Documents/data/hiwi_sample/tmp-mar...,left,94,8,2.00,1.000
84,Refined FT cluster,/home/maria/Documents/data/hiwi_sample/tmp-mar...,right,94,8,2.00,1.000


In [207]:
per_z_df[(per_z_df["hemisphere"] == "right") & (per_z_df["mask"] == "Original LC mask")]["voxel_count"].mean()

np.float64(11.25)

In [199]:
per_z_df[(per_z_df["hemisphere"] == "left") & (per_z_df["mask"] == "Refined FT cluster")]["voxel_count"].sum() #*(0.5*0.5*0.5)

np.int64(216)

In [158]:
### by peaking the max voxel within each region

In [229]:
lc_contrast_refined = compute_paper_lc_contrast(
    tse_path=MRI_IMAGE,
    lc_mask_path=LC_MASK_REFINED,
    reference_mask_path=DPT_MASK,
)

Peak z-index: 94
Left contrast:  0.2522
Right contrast: 0.3048
Final LC contrast: 0.2785 (27.85%)


In [234]:
lc_contrast_refined[["z_index", "left_peak_intensity", "right_peak_intensity", "reference_peak_intensity","left_contrast"]]

,z_index,left_peak_intensity,right_peak_intensity,reference_peak_intensity,left_contrast
0,88,496.711447,525.385458,496.570407,0.000284
1,89,490.063269,548.294767,502.095148,-0.023963
2,90,508.527993,526.578288,497.725909,0.021703
3,91,547.593840,532.356682,474.114291,0.154983
4,92,528.000666,560.874922,445.487023,0.185221
5,93,537.633751,568.402986,438.681945,0.225566
6,94,538.836818,561.446708,430.300188,0.252235
7,95,550.242097,561.426665,437.354253,0.258115
8,96,545.793960,546.136519,455.459530,0.198337
9,97,501.800298,530.979617,471.503563,0.064256


In [ ]:
### read folders and give the parameters

In [224]:
lc_contrast_original = compute_paper_lc_contrast(
    tse_path=MRI_IMAGE,
    lc_mask_path=LC_MASK,
    reference_mask_path=DPT_MASK,
)

Peak z-index: 95
Left contrast:  0.2489
Right contrast: 0.2581
Final LC contrast: 0.2535 (25.35%)


In [226]:
mean_contrast_refined, refined_per_slice_df = (
    compute_mean_lc_contrast(
        tse_path=MRI_IMAGE,
        lc_mask_path=LC_MASK_REFINED,
        reference_mask_path=DPT_MASK,
        return_per_slice=True,
        reference_max=True
    )
)

Peak z-index: 94
Left mean intensity: 483.0381
Right mean intensity: 525.5943
Reference peak intensity: 430.3002
Left contrast:  0.1226 (12.26%)
Right contrast: 0.2215 (22.15%)
Final mean LC contrast: 0.1720 (17.20%)


In [203]:
mean_contrast_refined, refined_per_slice_df = (
    compute_mean_lc_contrast(
        tse_path=MRI_IMAGE,
        lc_mask_path=LC_MASK_REFINED,
        reference_mask_path=DPT_MASK,
        return_per_slice=True,
        reference_max=False
    )
)

Peak z-index: 93
Left mean intensity: 484.9349
Right mean intensity: 521.5404
Reference peak intensity: 357.2888
Left contrast:  0.3573 (35.73%)
Right contrast: 0.4597 (45.97%)
Final mean LC contrast: 0.4085 (40.85%)


In [204]:
mean_contrast_refined, refined_per_slice_df = (
    compute_mean_lc_contrast(
        tse_path=MRI_IMAGE,
        lc_mask_path=LC_MASK,
        reference_mask_path=DPT_MASK,
        return_per_slice=True,
        reference_max=True
    )
)

Peak z-index: 94
Left mean intensity: 463.2272
Right mean intensity: 442.2062
Reference peak intensity: 430.3002
Left contrast:  0.0765 (7.65%)
Right contrast: 0.0277 (2.77%)
Final mean LC contrast: 0.0521 (5.21%)


In [205]:
mean_contrast_refined, refined_per_slice_df = (
    compute_mean_lc_contrast(
        tse_path=MRI_IMAGE,
        lc_mask_path=LC_MASK,
        reference_mask_path=DPT_MASK,
        return_per_slice=True,
        reference_max=False
    )
)

Peak z-index: 93
Left mean intensity: 456.0990
Right mean intensity: 441.7366
Reference peak intensity: 357.2888
Left contrast:  0.2766 (27.66%)
Right contrast: 0.2364 (23.64%)
Final mean LC contrast: 0.2565 (25.65%)
